# NeuroScan AI · train VGG16 + U-Net on a free GPU

Trains the two models the NeuroScan AI backend loads and produces four files:

| File | What it is |
|---|---|
| `vgg16_classifier.pt` | VGG16 (ImageNet pretrained) fine-tuned on 4 tumor classes |
| `vgg16_classifier.metrics.json` | accuracy, per-class precision/recall/F1, confusion matrix |
| `unet_vgg16.pt` | U-Net with a pretrained VGG16 encoder, trained on BraTS 2023 slices |
| `unet_vgg16.metrics.json` | Dice for whole tumor, tumor core, enhancing tumor |

**How to run**

1. In Colab: **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. **Runtime → Run all**. Total time is about 25–35 minutes on a T4.
3. A Google sign-in popup asks to connect Google Drive: allow it.
   Every checkpoint is written to **My Drive → NeuroScanAI → weights** as training runs,
   so nothing is lost if the browser tab closes or Colab disconnects.
4. At the end, `neuroscan_weights.zip` also downloads to your computer
   (and a copy stays in **My Drive → NeuroScanAI**).
5. Unzip it into `backend/weights/` of the project and restart the backend.

To skip Google Drive, set `SAVE_TO_DRIVE = False` in the first code cell; files then live only in the Colab session until downloaded.

Data (both free, no login): the Masoud Nickparvar Brain Tumor MRI dataset (CC BY 4.0, Hugging Face mirror) and BraTS 2023 Adult Glioma (Hugging Face `MedOtter/brats2023-gli-dataset`).

Research and education only. Not for clinical use.

In [ ]:
# Settings. The defaults suit a free Colab T4; raise BRATS_CASES / epochs if you have more time.
import os, sys, json, time, random, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NEUROSCAN_SMOKE") == "1"   # tiny run used to test this notebook on a CPU

WORK = Path(os.environ.get("NEUROSCAN_WORK", "/content"))
CLS_RAW = WORK / "nickparvar"
BRATS_RAW = WORK / "brats2023"
OUT = WORK / "neuroscan_weights"

# Save every checkpoint straight to Google Drive, so a dropped Colab session loses nothing.
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/NeuroScanAI"

CLS_EPOCHS = 1 if SMOKE else 10
CLS_PER_CLASS_LIMIT = 8 if SMOKE else None   # None = use every image
BRATS_CASES = 2 if SMOKE else 120            # of 1,251 available
UNET_EPOCHS = 1 if SMOKE else 15
SEG_BATCH = 2 if SMOKE else 8              # 8 fits a 16 GB T4 with mixed precision
SEED = 42

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel", "huggingface_hub"], check=True)
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")   # a Google sign-in popup appears: allow access
        OUT = Path(DRIVE_DIR) / "weights"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.models import VGG16_Weights, vgg16

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
OUT.mkdir(parents=True, exist_ok=True)
print("saving to:", OUT)
print("device:", DEVICE, torch.cuda.get_device_name(0) if AMP else "", "| smoke test" if SMOKE else "")
if not AMP and not SMOKE:
    print("WARNING: no GPU. Runtime → Change runtime type → T4 GPU, then run again.")

## Model definitions

Copied from `backend/app/models/` so the saved weights load straight into the backend. Do not rename layers.

In [ ]:
CLASS_NAMES = ("glioma", "meningioma", "no_tumor", "pituitary")
SEG_CLASSES = ("background", "necrotic_core", "edema", "enhancing")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def build_vgg16_classifier(num_classes=len(CLASS_NAMES), pretrained=True):
    model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    return model


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNetVGG16(nn.Module):
    def __init__(self, num_classes=len(SEG_CLASSES), pretrained_encoder=True):
        super().__init__()
        features = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained_encoder else None).features
        self.enc1 = features[0:4]
        self.enc2 = features[4:9]
        self.enc3 = features[9:16]
        self.enc4 = features[16:23]
        self.bottleneck = features[23:30]
        self.up4 = nn.ConvTranspose2d(512, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512 + 512, 256)
        self.up3 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256 + 256, 128)
        self.up2 = nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128 + 128, 64)
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64 + 64, 32)
        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        b = self.bottleneck(s4)
        d = self.dec4(torch.cat([self.up4(b), s4], 1))
        d = self.dec3(torch.cat([self.up3(d), s3], 1))
        d = self.dec2(torch.cat([self.up2(d), s2], 1))
        d = self.dec1(torch.cat([self.up1(d), s1], 1))
        return self.head(d)

print("models defined")

## 1 · Classifier data (7,200 MRI slices, 4 classes)

In [ ]:
from huggingface_hub import list_repo_files, snapshot_download

if not any(CLS_RAW.glob("Training/*/*.jpg")):
    t = time.time()
    snapshot_download("BJyotibrat/Masoud-Nickparvar-Brain-Tumor-MRI-Dataset", repo_type="dataset",
                      local_dir=CLS_RAW, allow_patterns=["Training/*", "Testing/*"], max_workers=16)
    print(f"downloaded in {time.time() - t:.0f}s")

def tf(train):
    steps = [transforms.Grayscale(3), transforms.Resize((224, 224))]
    if train:
        steps += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
                  transforms.ColorJitter(brightness=0.15, contrast=0.15)]
    return transforms.Compose(steps + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_full = datasets.ImageFolder(CLS_RAW / "Training", transform=tf(True))
train_eval = datasets.ImageFolder(CLS_RAW / "Training", transform=tf(False))
test_ds = datasets.ImageFolder(CLS_RAW / "Testing", transform=tf(False))
# Folder "notumor" sorts into the same slot as our "no_tumor".
assert [c.replace("notumor", "no_tumor") for c in train_full.classes] == list(CLASS_NAMES), train_full.classes

def limit(ds, n):
    if n is None:
        return list(range(len(ds)))
    seen, keep = {}, []
    for i, (_, y) in enumerate(ds.samples):
        if seen.get(y, 0) < n:
            keep.append(i); seen[y] = seen.get(y, 0) + 1
    return keep

# 10% of Training is held out for model selection; Testing is only used for the final report.
idx = limit(train_full, CLS_PER_CLASS_LIMIT)
random.Random(SEED).shuffle(idx)
n_val = max(4, len(idx) // 10)
val_idx, tr_idx = idx[:n_val], idx[n_val:]
test_idx = limit(test_ds, CLS_PER_CLASS_LIMIT)

workers = 2 if IN_COLAB else 0
cls_train = DataLoader(Subset(train_full, tr_idx), batch_size=32, shuffle=True, num_workers=workers, pin_memory=AMP)
cls_val = DataLoader(Subset(train_eval, val_idx), batch_size=64, num_workers=workers, pin_memory=AMP)
cls_test = DataLoader(Subset(test_ds, test_idx), batch_size=64, num_workers=workers, pin_memory=AMP)
print(f"train {len(tr_idx)} · val {len(val_idx)} · test {len(test_idx)}")

## 2 · Fine-tune VGG16

In [ ]:
def run_cls_epoch(model, loader, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    loss_sum = correct = seen = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=AMP):
                logits = model(x)
                loss = crit(logits, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            seen += len(y)
    return loss_sum / seen, correct / seen

clf = build_vgg16_classifier().to(DEVICE)
# Lower learning rate for the pretrained conv layers, higher for the classifier head.
opt = torch.optim.AdamW([
    {"params": clf.features.parameters(), "lr": 2e-5},
    {"params": clf.classifier.parameters(), "lr": 2e-4},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLS_EPOCHS)
scaler = torch.amp.GradScaler(enabled=AMP)

best_acc, cls_path = -1.0, OUT / "vgg16_classifier.pt"
for epoch in range(1, CLS_EPOCHS + 1):
    t = time.time()
    tr_loss, tr_acc = run_cls_epoch(clf, cls_train, opt, scaler)
    va_loss, va_acc = run_cls_epoch(clf, cls_val)
    sched.step()
    flag = ""
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({k: v.detach().cpu() for k, v in clf.state_dict().items()}, cls_path)
        flag = "  ← saved"
    print(f"epoch {epoch:02d}  train {tr_loss:.3f}/{tr_acc:.3f}  val {va_loss:.3f}/{va_acc:.3f}  {time.time() - t:.0f}s{flag}")

In [ ]:
# Final evaluation on the untouched Testing split, with the best checkpoint.
clf.load_state_dict(torch.load(cls_path, map_location=DEVICE, weights_only=True))
clf.eval()
n = len(CLASS_NAMES)
cm = [[0] * n for _ in range(n)]
with torch.inference_mode():
    for x, y in cls_test:
        pred = clf(x.to(DEVICE)).argmax(1).cpu()
        for t_, p_ in zip(y.tolist(), pred.tolist()):
            cm[t_][p_] += 1

per_class = {}
for i, name in enumerate(CLASS_NAMES):
    tp = cm[i][i]; fn = sum(cm[i]) - tp; fp = sum(cm[r][i] for r in range(n)) - tp
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    per_class[name] = {"precision": round(prec, 4), "recall": round(rec, 4),
                       "f1": round(2 * prec * rec / (prec + rec), 4) if prec + rec else 0.0, "support": sum(cm[i])}
total = sum(map(sum, cm))
cls_metrics = {
    "accuracy": round(sum(cm[i][i] for i in range(n)) / total, 4),
    "classes": list(CLASS_NAMES),
    "confusion_matrix": cm,
    "per_class": per_class,
    "val_images": total,
    "mode": "full fine-tune (Colab GPU)",
    "epochs": CLS_EPOCHS,
    "best_val_acc_during_training": round(best_acc, 4),
}
(OUT / "vgg16_classifier.metrics.json").write_text(json.dumps(cls_metrics, indent=2))
print(f"TEST ACCURACY {cls_metrics['accuracy']:.4f}")
for k, v in per_class.items():
    print(f"  {k:<11} P {v['precision']:.3f}  R {v['recall']:.3f}  F1 {v['f1']:.3f}")

## 3 · Segmentation data (BraTS 2023 3D volumes, sliced to 2D)

In [ ]:
import nibabel as nib

REPO = "MedOtter/brats2023-gli-dataset"
local_cases = sorted({p.parent.name for p in BRATS_RAW.rglob("*-seg.nii.gz")})
if len(local_cases) < BRATS_CASES:
    files = list_repo_files(REPO, repo_type="dataset")
    cases = sorted({f.split("/")[1] for f in files if f.endswith("-seg.nii.gz")})[:BRATS_CASES]
    t = time.time()
    snapshot_download(REPO, repo_type="dataset", local_dir=BRATS_RAW, max_workers=16,
                      allow_patterns=[f"*/{c}/{c}-{m}.nii.gz" for c in cases for m in ("t1c", "seg")])
    print(f"downloaded {len(cases)} cases in {time.time() - t:.0f}s")

def normalise(vol):
    brain = vol[vol > 0]
    if brain.size == 0:
        return np.zeros_like(vol, dtype=np.uint8)
    lo, hi = np.percentile(brain, (1, 99))
    return (np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)

SIZE = 256

def load_case(seg_path):
    case = seg_path.name.removesuffix("-seg.nii.gz")
    img = normalise(np.asarray(nib.load(str(seg_path.with_name(f"{case}-t1c.nii.gz"))).dataobj, dtype=np.float32))
    seg = np.asarray(nib.load(str(seg_path)).dataobj).astype(np.uint8)
    tumor = [z for z in range(seg.shape[2]) if seg[:, :, z].any()]
    empty = [z for z in range(seg.shape[2]) if img[:, :, z].max() > 10 and not seg[:, :, z].any()]
    keep = tumor + random.Random(case).sample(empty, min(len(empty), len(tumor) // 6))
    xs, ys = [], []
    for z in keep:
        # Same orientation and resizing as the backend (rot90 of the axial slice).
        xs.append(np.asarray(Image.fromarray(np.rot90(img[:, :, z]).copy()).resize((SIZE, SIZE), Image.BILINEAR)))
        ys.append(np.asarray(Image.fromarray(np.rot90(seg[:, :, z]).copy()).resize((SIZE, SIZE), Image.NEAREST)))
    return np.stack(xs), np.stack(ys)

seg_paths = sorted(BRATS_RAW.rglob("*-seg.nii.gz"))[:BRATS_CASES]
random.Random(SEED).shuffle(seg_paths)
n_val = max(1, round(len(seg_paths) * 0.2))
splits = {"val": seg_paths[:n_val], "train": seg_paths[n_val:]}
arrays = {}
for split, paths in splits.items():
    parts = [load_case(p) for p in paths]
    arrays[split] = (np.concatenate([a for a, _ in parts]), np.concatenate([b for _, b in parts]))
    print(f"{split}: {len(paths)} patients, {len(arrays[split][0])} slices")

class SliceSet(Dataset):
    def __init__(self, x, y, augment):
        self.x, self.y, self.augment = x, y, augment
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        img = self.x[i].astype(np.float32) / 255.0
        mask = self.y[i].astype(np.int64)
        if self.augment and random.random() < 0.5:
            img, mask = img[:, ::-1].copy(), mask[:, ::-1].copy()
        rgb = (np.stack([img] * 3, 0) - np.array(MEAN, np.float32)[:, None, None]) / np.array(STD, np.float32)[:, None, None]
        return torch.from_numpy(rgb.astype(np.float32)), torch.from_numpy(mask)

seg_train = DataLoader(SliceSet(*arrays["train"], True), batch_size=SEG_BATCH, shuffle=True, num_workers=workers, pin_memory=AMP)
seg_val = DataLoader(SliceSet(*arrays["val"], False), batch_size=SEG_BATCH * 2, num_workers=workers, pin_memory=AMP)

## 4 · Train the U-Net

In [ ]:
REGIONS = {"whole_tumor": (1, 2, 3), "tumor_core": (1, 3), "enhancing_tumor": (3,)}

def dice_loss(logits, target, eps=1e-6):
    probs = F.softmax(logits.float(), 1)
    onehot = F.one_hot(target, len(SEG_CLASSES)).permute(0, 3, 1, 2).float()
    inter = (probs * onehot).sum((0, 2, 3))
    denom = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    return 1 - ((2 * inter + eps) / (denom + eps))[1:].mean()

def run_seg_epoch(model, loader, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    ce = nn.CrossEntropyLoss()
    total = seen = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=AMP):
                logits = model(x)
                loss = ce(logits, y) + dice_loss(logits, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            total += loss.item() * len(x); seen += len(x)
    return total / seen

unet = UNetVGG16().to(DEVICE)
encoder = [p for n_, p in unet.named_parameters() if n_.split(".")[0] in ("enc1", "enc2", "enc3", "enc4", "bottleneck")]
decoder = [p for n_, p in unet.named_parameters() if n_.split(".")[0] not in ("enc1", "enc2", "enc3", "enc4", "bottleneck")]
opt = torch.optim.AdamW([{"params": encoder, "lr": 2e-5}, {"params": decoder, "lr": 3e-4}], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=UNET_EPOCHS)
scaler = torch.amp.GradScaler(enabled=AMP)

best_loss, seg_path = float("inf"), OUT / "unet_vgg16.pt"
for epoch in range(1, UNET_EPOCHS + 1):
    t = time.time()
    tr = run_seg_epoch(unet, seg_train, opt, scaler)
    va = run_seg_epoch(unet, seg_val)
    sched.step()
    flag = ""
    if va < best_loss:
        best_loss = va
        torch.save({k: v.detach().cpu() for k, v in unet.state_dict().items()}, seg_path)
        flag = "  ← saved"
    print(f"epoch {epoch:02d}  train {tr:.4f}  val {va:.4f}  {time.time() - t:.0f}s{flag}")

In [ ]:
# Dice per BraTS region on held-out patients, with the best checkpoint.
unet.load_state_dict(torch.load(seg_path, map_location=DEVICE, weights_only=True))
unet.eval()
inter = {k: 0 for k in REGIONS}; denom = {k: 0 for k in REGIONS}
with torch.inference_mode():
    for x, y in seg_val:
        pred = unet(x.to(DEVICE)).argmax(1).cpu()
        for k, labels in REGIONS.items():
            lab = torch.tensor(labels)
            p, g = torch.isin(pred, lab), torch.isin(y, lab)
            inter[k] += (p & g).sum().item(); denom[k] += p.sum().item() + g.sum().item()
seg_metrics = {
    "dice": {k: round(2 * inter[k] / denom[k], 4) if denom[k] else None for k in REGIONS},
    "best_val_loss": round(best_loss, 4),
    "epochs": UNET_EPOCHS,
    "freeze_encoder": False,
    "train_slices": len(seg_train.dataset),
    "val_slices": len(seg_val.dataset),
}
(OUT / "unet_vgg16.metrics.json").write_text(json.dumps(seg_metrics, indent=2))
print("DICE", seg_metrics["dice"])

## 5 · Check and download

In [ ]:
import shutil

# Reload both files the way the backend does (strict key match, weights_only).
m = build_vgg16_classifier(pretrained=False)
m.load_state_dict(torch.load(OUT / "vgg16_classifier.pt", map_location="cpu", weights_only=True))
u = UNetVGG16(pretrained_encoder=False)
u.load_state_dict(torch.load(OUT / "unet_vgg16.pt", map_location="cpu", weights_only=True))
with torch.inference_mode():
    assert m.eval()(torch.zeros(1, 3, 224, 224)).shape == (1, 4)
    assert u.eval()(torch.zeros(1, 3, 256, 256)).shape == (1, 4, 256, 256)

archive = shutil.make_archive(str(OUT.parent / "neuroscan_weights"), "zip", OUT)
for f in sorted(OUT.iterdir()):
    print(f"{f.name:<32} {f.stat().st_size / 1e6:7.1f} MB")
print("archive:", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)

## Install on your computer

If the browser download did not start, open **My Drive → NeuroScanAI** and download `neuroscan_weights.zip` from there.

```bash
cd "Longitutdinal Brain Tumour Segmentation/backend"
unzip -o ~/Downloads/neuroscan_weights.zip -d weights/
```

Restart the backend (stop `uvicorn` and start it again). The top bar of the site shows the API online, and the Model page shows the accuracy, confusion matrix and Dice from this run.